# Lesson 7 | The first RTL neuron

Last lesson introduced **Register-Transfer Level (RTL)**. Today asks:
> **How is one neuron update split into a combinational path and a sequential register update?**

To avoid choosing unfrozen LIF/fixed-point details, this lesson uses a teaching integrate-and-fire neuron that reuses Lesson 4's accumulator + threshold + reset contract. It is not formal `MOD-003`.


## 1. Concept ledger

**Known:** register, clock edge, RTL module/port, and one simple `always_ff` example.

**New:** combinational path, `always_comb`, and how it works with sequential `always_ff`.

**Preview:** testbench and waveform wait until next lesson.


## 2. Lesson contract

Each cycle: read old `membrane_v` → `candidate = membrane_v + input_current` → `candidate >= threshold` produces a spike condition → choose reset or candidate for next state → store it at the clock edge.

This lesson fixes an 8-bit signed demo width and deliberately avoids overflow vectors. Formal rounding, saturation, and leak still belong to RMD-002 / RMD-003.


## 3. Replay the rule with a Python oracle

Predict spike cycles for threshold 4 and inputs `[1,1,1,1,2,2]`, then run.


In [ ]:
def tutorial_if_step(state, input_current, threshold, reset_value=0):
    candidate = state + input_current
    spike = candidate >= threshold
    next_state = reset_value if spike else candidate
    return next_state, spike, candidate

state = 0
for cycle, current in enumerate([1, 1, 1, 1, 2, 2]):
    next_state, spike, candidate = tutorial_if_step(state, current, 4)
    print(f'cycle={cycle}: state={state}, input={current}, candidate={candidate}, spike={spike}, next={next_state}')
    state = next_state


## 4. Structure

```mermaid
flowchart LR
 REG["membrane_v register"] --> ADD["adder"]
 IN["input_current"] --> ADD
 ADD --> C["candidate"]
 C --> CMP[">= threshold"]
 TH["threshold"] --> CMP
 CMP --> SEL["choose reset or candidate"]
 C --> SEL
 R["reset_value"] --> SEL
 SEL --> NEXT["next_v"]
 NEXT --> REG
 CLK["clock edge"] -.-> REG
```


## 5. `always_comb` and `always_ff`

`always_comb` describes next-state logic that stores no history; `always_ff @(posedge clk)` stores results at the edge.

The first RTL neuron intentionally uses a fixed 8-bit width rather than adding parameterization, explicit sign extension, or the formal overflow policy all at once.


## 6. Complete teaching RTL

```systemverilog
module tutorial_if_neuron (
    input  logic              clk,
    input  logic              rst_n,
    input  logic signed [7:0] input_current,
    input  logic signed [7:0] threshold,
    input  logic signed [7:0] reset_value,
    output logic signed [7:0] membrane_v,
    output logic              spike
);
    logic signed [7:0] candidate;
    logic signed [7:0] next_v;
    logic spike_next;

    always_comb begin
        candidate = membrane_v + input_current;
        spike_next = candidate >= threshold;

        if (spike_next)
            next_v = reset_value;
        else
            next_v = candidate;
    end

    always_ff @(posedge clk) begin
        if (!rst_n) begin
            membrane_v <= reset_value;
            spike <= 1'b0;
        end else begin
            membrane_v <= next_v;
            spike <= spike_next;
        end
    end
endmodule
```


## 7. Read it in blocks

`candidate`, `spike_next`, and `next_v` are combinational results of current state/input. `membrane_v` and `spike` update after the clock edge.

`if (spike_next)` describes a hardware selection between reset and candidate; it is not a CPU dynamically deciding whether to create a mux.


### Optional experiment: how can 8-bit overflow break intuition?

The teaching RTL stores `candidate` in an 8-bit signed value with range `-128..127`. If the mathematical result exceeds that range, hardware keeps only the finite-width bit pattern; interpreting it as two's complement can turn the result negative.

Predict what `100 + 50` becomes as an 8-bit signed value, then run the Python experiment below.

In [ ]:
def wrap_signed_8(x: int) -> int:
    return ((x + 128) % 256) - 128

mathematical_sum = 100 + 50
wrapped_sum = wrap_signed_8(mathematical_sum)
print('mathematical sum =', mathematical_sum)
print('8-bit signed result =', wrapped_sum)


The result is `150 -> -106`. With a positive threshold, the current teaching RTL could therefore fail to spike, which is exactly why finite-width arithmetic needs an explicit policy.

**Do not turn this observation into a formal golden test.** Overflow is intentionally outside this lesson contract. Wrap-around is a property of the current 8-bit teaching implementation, not an approved neuron semantic. Formal RTL must return to RMD-002 / RMD-003, choose widening, saturation, wrap-around, or another policy explicitly, and only then freeze the corresponding test oracle.

A second parameter boundary is worth predicting: if `reset_value >= threshold` and the next-cycle input is zero, the reset state already satisfies the threshold condition, so this teaching module will continue to spike on subsequent cycles. That is not a bug to “fix” in this lesson; it follows directly from the chosen parameters. The boundary testbench characterizes it as a teaching observation, while a later model specification decides whether such parameter combinations are allowed.

## 8. Try It: work one edge by hand

Before an edge: `membrane_v=3, input_current=1, threshold=4, reset_value=0`. Write candidate / spike_next / next_v, then post-edge membrane_v / spike.


## 9. AI Task

Ask AI to map each of the five contract rules to the exact RTL statements. If it finds a mismatch, report it rather than changing the contract.


## 10. Human Check

Without AI, identify the combinational path and stored state, explain `always_comb` vs `always_ff`, explain how `candidate=4` and post-edge `membrane_v=0` can both be correct, and explain why this teaching module is not formal LIF RTL.


## 11. Engineering Handoff

`rtl/learning/tutorial_if_neuron.sv` demonstrates mapping a known contract into RTL structure. Formal `rtl/neuron/lif_neuron_engine.sv` still waits for v0 semantics and the fixed-point contract.


## 12. Project Trace

- Lesson: `LSN-007`
- Mapping: `RMD-004` teaching precursor
- Formal `MOD-003`: intentionally not created


## 13. Exit Ticket

You can derive a combinational-path-plus-register structure from the contract and identify both parts in RTL. Next lesson adds no neuron behavior; it verifies this one.
